## 1_3_1 Saccade Detection + Quantification (rebuild)

This notebook currently includes only:
- input contract + loading of selected eye parquet from `1_2` metadata
- detection + k-sweep parameter block
- preprocessing (`X_raw`, `X_smooth`, `vel_x_smooth`)
- adaptive single-pass `k` sweep with heuristic selection
- final detection run at selected `k`
- optional debug plot of `Ellipse.Center.X`


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.signal import find_peaks
from plotly.subplots import make_subplots

In [ ]:
# Cell 1: Input contract + load selected eye data from 1_2 metadata
##########################################################################

# good S/N
data_path = Path(
    "/Users/rancze/Documents/Data/vestVR/20250409_Cohort3_rotation/Visual_mismatch_day4/B6J2783-2025-04-28T14-57-30"
)

# #bad S/N
# data_path = Path(
#     "/Users/rancze/Documents/Data/vestVR/20250409_Cohort3_rotation/Visual_mismatch_day4/B6J2782-2025-04-28T14-22-03"
# )

# data_path = Path(
#     "/Users/rancze/Documents/Data/vestVR/20250409_Cohort3_rotation/Visual_mismatch_day4/B6J2781-2025-04-28T13-45-40"
# )


debug = False

save_path = data_path.parent / f"{data_path.name}_processedData"
downsampled_output_dir = save_path / "downsampled_data"
metadata_path = downsampled_output_dir / "saccade_input_metadata.json"

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Metadata not found at {metadata_path}. Run 1_2_SLEAP_processing.ipynb first."
    )

with open(metadata_path, "r") as f:
    metadata = json.load(f)

required_top_keys = [
    "eye_with_least_low_confidence",
    "video1_eye",
    "video2_eye",
    "outputs",
]
missing_top_keys = [k for k in required_top_keys if k not in metadata]
if missing_top_keys:
    raise KeyError(f"Metadata missing required keys: {missing_top_keys}")

eye_with_least_low_confidence = metadata.get("eye_with_least_low_confidence")
if eye_with_least_low_confidence not in {"VideoData1", "VideoData2"}:
    raise ValueError(
        "metadata['eye_with_least_low_confidence'] is missing or invalid. "
        "Expected 'VideoData1' or 'VideoData2'."
    )

selected_eye_code = (
    metadata.get("video1_eye")
    if eye_with_least_low_confidence == "VideoData1"
    else metadata.get("video2_eye")
)
if selected_eye_code not in {"L", "R"}:
    selected_eye_code = "Unknown"

video_outputs = metadata.get("outputs", {}).get(eye_with_least_low_confidence, {})
video_parquet_path = video_outputs.get("resampled_parquet")
if video_parquet_path is None:
    raise ValueError(
        f"No resampled_parquet path found in metadata for {eye_with_least_low_confidence}."
    )

video_parquet_path = Path(video_parquet_path)
if not video_parquet_path.exists():
    raise FileNotFoundError(
        f"Parquet file not found at {video_parquet_path} for {eye_with_least_low_confidence}."
    )

VideoData = pd.read_parquet(video_parquet_path).reset_index(drop=True)

required_columns = ["Seconds", "Ellipse.Center.X"]
missing_columns = [c for c in required_columns if c not in VideoData.columns]
if missing_columns:
    raise KeyError(f"Loaded VideoData is missing required columns: {missing_columns}")

if VideoData["Seconds"].diff().dropna().le(0).any():
    raise ValueError("Seconds must be strictly increasing.")

if "frame_idx" not in VideoData.columns:
    VideoData["frame_idx"] = np.arange(len(VideoData), dtype=int)

if debug:
    print(f"Loaded metadata: {metadata_path}")
    print(f"Selected eye/video: {eye_with_least_low_confidence} ({selected_eye_code})")
    print(f"Loaded parquet: {video_parquet_path}")
    print(f"VideoData shape: {VideoData.shape}")
    display(VideoData.head())

    # Debug plot (inline): Ellipse.Center.X vs relative time
    time_rel_s = VideoData["Seconds"] - VideoData["Seconds"].iloc[0]
    fig = go.Figure(
        data=[
            go.Scatter(
                x=time_rel_s,
                y=VideoData["Ellipse.Center.X"],
                mode="lines",
                name="Ellipse.Center.X",
                line=dict(width=1),
            )
        ]
    )
    fig.update_layout(
        title=f"Ellipse.Center.X ({eye_with_least_low_confidence}, eye={selected_eye_code})",
        xaxis_title="Relative time (s)",
        yaxis_title="Ellipse.Center.X (px)",
        template="plotly_white",
    )
    fig.show()
else:
    print(
        f"✅ Loaded selected eye/video: {eye_with_least_low_confidence} ({selected_eye_code})"
    )

In [ ]:
# Cell 2: Detection and k-sweep parameters
##########################################################################

# mostly tweek:
# k_overdetect_offset for overdetection (often needed high for transient detection)
# for transient removal: refractory_period_s, transient_pair_max_net_displacement_px
# for amplitude filtering: min_saccade_amplitude_px

# --- Signal preprocessing ---
smoothing_window_s = 0.08  # median smoothing window before velocity calculation

# --- Velocity-threshold event detection ---
peak_width_time_s = 0.005  # minimum peak width for velocity peaks
onset_fraction = (
    0.2  # for saccade duration, start boundary when |vel| falls below this * threshold
)
offset_fraction = (
    0.1  # for saccade duration, end boundary when |vel| falls below this * threshold
)

# --- k-sweep search space + scoring ---
fine_step = 0.1  # k-grid resolution
k_hard_min = 2.0  # lower bound for tested/selected k
k_hard_max = 4.0  # upper bound for tested/selected k
overdetect_bias_strength = 1  # >0 favors slightly lower k (more detections)
k_overdetect_offset = (
    -0.25
)  # extra push toward lower k after heuristic selection for strong overdetection
stability_guardrail_drop = 5.0  # max allowed stability loss after k offset
k_fallback = 4  # legacy fallback if sweep cannot produce a valid suggestion

# --- Post-detection filtering ---
refractory_period_s = 0.2  # to detect transients, transient-pair ISI window (not used in find_peaks spacing)
same_direction_dedup_window_s = (
    refractory_period_s  # collapse close same-direction duplicates
)
transient_pair_max_net_displacement_px = 5.0  # max pair net displacement to classify close opposite-direction transient, i.e. does the transient come back to baseline?
min_saccade_amplitude_px = 4.0  # minimum amplitude to keep final event

# --- Optional downstream analysis/QC controls ---
pre_window_s = 0.15  # peri-event snippet window before event time
post_window_s = 0.5  # peri-event snippet window after event time
baseline_window_start_s = -0.06  # baseline window start (relative to event)
baseline_window_end_s = -0.02  # baseline window end (relative to event)
plot_detection_qc = True  # render Cell 8/9 QC figures

# If metadata already has saved parameters, override defaults from this cell.
loaded_params = metadata.get("saccade_detection_parameters", {})
if isinstance(loaded_params, dict) and len(loaded_params) > 0:

    def _as_bool(value):
        if isinstance(value, bool):
            return value
        if isinstance(value, (int, float)):
            return bool(value)
        if isinstance(value, str):
            return value.strip().lower() in {"1", "true", "yes", "y", "on"}
        return bool(value)

    def _param_or_default(key, default, caster):
        raw_value = loaded_params.get(key, default)
        try:
            return caster(raw_value)
        except (TypeError, ValueError):
            return default

    # Signal preprocessing
    smoothing_window_s = _param_or_default(
        "smoothing_window_s", smoothing_window_s, float
    )
    # Velocity-threshold event detection
    peak_width_time_s = _param_or_default("peak_width_time_s", peak_width_time_s, float)
    onset_fraction = _param_or_default("onset_fraction", onset_fraction, float)
    offset_fraction = _param_or_default("offset_fraction", offset_fraction, float)
    # k-sweep search space + scoring
    fine_step = _param_or_default("fine_step", fine_step, float)
    k_hard_min = _param_or_default("k_hard_min", k_hard_min, float)
    k_hard_max = _param_or_default("k_hard_max", k_hard_max, float)
    overdetect_bias_strength = _param_or_default(
        "overdetect_bias_strength", overdetect_bias_strength, float
    )
    k_overdetect_offset = _param_or_default(
        "k_overdetect_offset", k_overdetect_offset, float
    )
    stability_guardrail_drop = _param_or_default(
        "stability_guardrail_drop", stability_guardrail_drop, float
    )
    k_fallback = _param_or_default("k_fallback", k_fallback, float)
    # Post-detection filtering
    refractory_period_s = _param_or_default(
        "refractory_period_s", refractory_period_s, float
    )
    same_direction_dedup_window_s = _param_or_default(
        "same_direction_dedup_window_s", same_direction_dedup_window_s, float
    )
    transient_pair_max_net_displacement_px = _param_or_default(
        "transient_pair_max_net_displacement_px",
        transient_pair_max_net_displacement_px,
        float,
    )
    min_saccade_amplitude_px = _param_or_default(
        "min_saccade_amplitude_px", min_saccade_amplitude_px, float
    )
    # Optional downstream analysis/QC controls
    pre_window_s = _param_or_default("pre_window_s", pre_window_s, float)
    post_window_s = _param_or_default("post_window_s", post_window_s, float)
    baseline_window_start_s = _param_or_default(
        "baseline_window_start_s", baseline_window_start_s, float
    )
    baseline_window_end_s = _param_or_default(
        "baseline_window_end_s", baseline_window_end_s, float
    )
    plot_detection_qc = _param_or_default("plot_detection_qc", plot_detection_qc, _as_bool)

    print("ℹ️ Loaded Cell 2 parameter overrides from metadata.")

print("✅ Detection and k-sweep parameters initialized")

In [ ]:
# Cell 2b: Save Cell 2 parameters into metadata
##########################################################################
cell2_params = {
    # Signal preprocessing
    "smoothing_window_s": float(smoothing_window_s),
    # Velocity-threshold event detection
    "peak_width_time_s": float(peak_width_time_s),
    "onset_fraction": float(onset_fraction),
    "offset_fraction": float(offset_fraction),
    # k-sweep search space + scoring
    "fine_step": float(fine_step),
    "k_hard_min": float(k_hard_min),
    "k_hard_max": float(k_hard_max),
    "overdetect_bias_strength": float(overdetect_bias_strength),
    "k_overdetect_offset": float(k_overdetect_offset),
    "stability_guardrail_drop": float(stability_guardrail_drop),
    "k_fallback": float(k_fallback),
    # Post-detection filtering
    "refractory_period_s": float(refractory_period_s),
    "same_direction_dedup_window_s": float(same_direction_dedup_window_s),
    "transient_pair_max_net_displacement_px": float(
        transient_pair_max_net_displacement_px
    ),
    "min_saccade_amplitude_px": float(min_saccade_amplitude_px),
    # Optional downstream analysis/QC controls
    "pre_window_s": float(pre_window_s),
    "post_window_s": float(post_window_s),
    "baseline_window_start_s": float(baseline_window_start_s),
    "baseline_window_end_s": float(baseline_window_end_s),
    "plot_detection_qc": bool(plot_detection_qc),
}

metadata["saccade_detection_parameters"] = cell2_params
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Saved Cell 2 parameters to metadata: {metadata_path}")

In [ ]:
# Cell 3: Preprocess eye position for velocity-based saccade detection
##########################################################################

df_work = VideoData[["Seconds", "Ellipse.Center.X", "frame_idx"]].copy()
df_work["X_raw"] = df_work["Ellipse.Center.X"].astype(float)

fps = 1.0 / df_work["Seconds"].diff().median()

smoothing_window_points = max(1, int(round(smoothing_window_s * fps)))
df_work["X_smooth"] = (
    df_work["X_raw"]
    .rolling(window=smoothing_window_points, center=True)
    .median()
    .bfill()
    .ffill()
)

df_work["dt"] = df_work["Seconds"].diff()
eps = np.finfo(float).eps
df_work.loc[df_work["dt"].abs() <= eps, "dt"] = np.nan
df_work.loc[df_work["dt"] <= 0, "dt"] = np.nan

df_work["vel_x_smooth"] = (df_work["X_smooth"].diff() / df_work["dt"]).astype(float)
df_work["vel_x_raw"] = (df_work["X_raw"].diff() / df_work["dt"]).astype(float)
df_work["vel_x_smooth"] = df_work["vel_x_smooth"].where(
    np.isfinite(df_work["vel_x_smooth"]), np.nan
)
df_work["vel_x_raw"] = df_work["vel_x_raw"].where(
    np.isfinite(df_work["vel_x_raw"]), np.nan
)

print(
    f"✅ Preprocessing complete | fps={fps:.3f} | smoothing_window_points={smoothing_window_points}"
)

In [ ]:
# Cell 4: Prepare shared velocity summary for k-sweep
##########################################################################
# Keep one shared velocity summary for sweeps
abs_vel = df_work["vel_x_smooth"].abs().dropna()
if len(abs_vel) == 0:
    raise ValueError("No finite velocity samples available for saccade detection.")
print("ℹ️ Velocity summary prepared for k-sweep detection workflow.")

In [ ]:
# Cell 5: Single-pass k-value sweep for adaptive threshold tuning
##########################################################################
k_sweep_values = np.arange(2.5, 7.0 + 1e-9, fine_step)

# Enforce hard bounds for sweep grids.
k_sweep_values = k_sweep_values[
    (k_sweep_values >= k_hard_min) & (k_sweep_values <= k_hard_max)
]

v = df_work["vel_x_smooth"].to_numpy()
x = df_work["X_raw"].to_numpy()
t = df_work["Seconds"].to_numpy()
f = df_work["frame_idx"].to_numpy()
peak_width_points = max(1, int(round(peak_width_time_s * fps)))


def _split_transient_pairs(
    events_df: pd.DataFrame, max_isi_s: float, max_net_displacement_px: float
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split events into kept vs transient-pair filtered (dropped) events."""
    if len(events_df) < 2:
        return events_df.reset_index(drop=True), pd.DataFrame(columns=events_df.columns)

    times = events_df["time"].to_numpy(dtype=float)
    directions = events_df["direction"].to_numpy()
    starts = events_df["start_position"].to_numpy(dtype=float)
    ends = events_df["end_position"].to_numpy(dtype=float)
    keep_mask = np.ones(len(events_df), dtype=bool)

    i = 0
    while i < len(events_df) - 1:
        isi = times[i + 1] - times[i]
        opposite_direction = directions[i] != directions[i + 1]
        pair_net_disp = abs(ends[i + 1] - starts[i])
        if (
            isi <= max_isi_s
            and opposite_direction
            and pair_net_disp <= max_net_displacement_px
        ):
            keep_mask[i] = False
            keep_mask[i + 1] = False
            i += 2
            continue
        i += 1

    kept_df = events_df.loc[keep_mask].reset_index(drop=True)
    dropped_df = events_df.loc[~keep_mask].reset_index(drop=True)
    return kept_df, dropped_df


def _dedupe_same_direction_events(
    events_df: pd.DataFrame, min_isi_s: float
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Keep strongest event within close same-direction clusters."""
    if len(events_df) < 2:
        return events_df.reset_index(drop=True), pd.DataFrame(columns=events_df.columns)

    times = events_df["time"].to_numpy(dtype=float)
    directions = events_df["direction"].to_numpy()
    peak_abs_vel = np.abs(events_df["velocity"].to_numpy(dtype=float))
    keep_mask = np.ones(len(events_df), dtype=bool)

    i = 0
    while i < len(events_df):
        cluster_indices = [i]
        j = i + 1
        while (
            j < len(events_df)
            and directions[j] == directions[j - 1]
            and (times[j] - times[j - 1]) <= min_isi_s
        ):
            cluster_indices.append(j)
            j += 1

        if len(cluster_indices) > 1:
            strongest_idx = cluster_indices[
                int(np.argmax(peak_abs_vel[cluster_indices]))
            ]
            for idx in cluster_indices:
                if idx != strongest_idx:
                    keep_mask[idx] = False

        i = j

    kept_df = events_df.loc[keep_mask].reset_index(drop=True)
    dropped_df = events_df.loc[~keep_mask].reset_index(drop=True)
    return kept_df, dropped_df


def _events_for_k(k_value: float) -> tuple[pd.DataFrame, float, float, float]:
    """Run detection for a single k and return events + threshold stats."""
    vel_mean_k = abs_vel.mean()
    vel_std_k = abs_vel.std()
    vel_thresh_k = vel_mean_k + k_value * vel_std_k

    pos_peaks_k, _ = find_peaks(
        v,
        height=vel_thresh_k,
        width=peak_width_points,
    )
    neg_peaks_k, _ = find_peaks(
        -v,
        height=vel_thresh_k,
        width=peak_width_points,
    )

    def _extract_events_k(peak_indices, direction):
        events = []
        for peak_idx in peak_indices:
            start_idx = int(peak_idx)
            end_idx = int(peak_idx)

            while start_idx > 0 and abs(v[start_idx]) > vel_thresh_k * onset_fraction:
                start_idx -= 1

            while (
                end_idx < len(v) - 1
                and abs(v[end_idx]) > vel_thresh_k * offset_fraction
            ):
                end_idx += 1
            if end_idx < len(v) - 1:
                end_idx += 1

            start_x = x[start_idx]
            end_x = x[end_idx]
            displacement = end_x - start_x

            events.append(
                {
                    "direction": direction,
                    "time": float(t[peak_idx]),
                    "velocity": float(v[peak_idx]),
                    "start_time": float(t[start_idx]),
                    "end_time": float(t[end_idx]),
                    "duration": float(t[end_idx] - t[start_idx]),
                    "start_position": float(start_x),
                    "end_position": float(end_x),
                    "amplitude": float(abs(displacement)),
                    "displacement": float(displacement),
                    "start_frame_idx": int(f[start_idx]),
                    "peak_frame_idx": int(f[peak_idx]),
                    "end_frame_idx": int(f[end_idx]),
                }
            )
        return pd.DataFrame(events)

    up_df = _extract_events_k(pos_peaks_k, "upward")
    down_df = _extract_events_k(neg_peaks_k, "downward")
    events_df = pd.concat([up_df, down_df], ignore_index=True)
    if len(events_df) > 0:
        events_df = events_df.sort_values("time").reset_index(drop=True)
    return events_df, vel_thresh_k, vel_mean_k, vel_std_k


def _evaluate_k_grid(k_values: np.ndarray, label: str) -> pd.DataFrame:
    if len(k_values) == 0:
        return pd.DataFrame(
            columns=[
                "sweep",
                "k",
                "vel_thresh",
                "vel_mean",
                "vel_std",
                "n_saccades",
                "median_duration",
                "median_amplitude",
                "stability_score",
            ]
        )

    rows = []
    for kval in k_values:
        events_df, thr, m, s = _events_for_k(float(kval))
        n = len(events_df)
        rows.append(
            {
                "sweep": label,
                "k": float(kval),
                "vel_thresh": float(thr),
                "vel_mean": float(m),
                "vel_std": float(s),
                "n_saccades": int(n),
                "median_duration": float(events_df["duration"].median())
                if n > 0
                else np.nan,
                "median_amplitude": float(events_df["amplitude"].median())
                if n > 0
                else np.nan,
            }
        )

    df_metrics = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)

    # Stability across neighboring k values based on count-change percentage.
    n_vals = df_metrics["n_saccades"].astype(float).to_numpy()
    local_change_pct = []
    for i in range(len(df_metrics)):
        changes = []
        if i > 0 and n_vals[i - 1] > 0:
            changes.append(abs(n_vals[i] - n_vals[i - 1]) / n_vals[i - 1] * 100.0)
        if i < len(df_metrics) - 1 and n_vals[i + 1] > 0:
            changes.append(
                abs(n_vals[i + 1] - n_vals[i]) / n_vals[i] * 100.0
                if n_vals[i] > 0
                else np.nan
            )
        local_change_pct.append(
            float(np.nanmean(changes)) if len(changes) > 0 else np.nan
        )

    # Only keep stability_score (derived from neighboring count consistency).
    df_metrics["stability_score"] = np.clip(
        100.0 - np.array(local_change_pct), 0.0, 100.0
    )
    return df_metrics


def _add_composite_score(df_metrics: pd.DataFrame) -> pd.DataFrame:
    """Add selection heuristic score with slight over-detection bias."""
    if len(df_metrics) < 3:
        df_metrics["composite_score"] = np.nan
        return df_metrics

    norm = lambda s: (s - s.mean()) / (s.std(ddof=0) + 1e-9)
    composite_score = (
        norm(df_metrics["stability_score"])
        + overdetect_bias_strength * norm(df_metrics["n_saccades"])
        - 0.25 * norm(df_metrics["k"])
    )
    df_metrics["composite_score"] = composite_score
    return df_metrics


sweep_metrics_df = _evaluate_k_grid(k_sweep_values, label="sweep")
sweep_metrics_df = _add_composite_score(sweep_metrics_df)
print("✅ k sweep complete")
if debug:
    display(sweep_metrics_df)

# Suggested k from single sweep
if len(sweep_metrics_df) >= 3 and not sweep_metrics_df["composite_score"].isna().all():
    k_suggested = float(
        sweep_metrics_df.loc[sweep_metrics_df["composite_score"].idxmax(), "k"]
    )
else:
    k_suggested = float(k_fallback)

print(f"🎯 Suggested k from sweep: {k_suggested:.2f}")

# Apply absolute over-detection offset with stability guardrail.
if len(sweep_metrics_df) > 0 and sweep_metrics_df["stability_score"].notna().any():
    sweep_k_vals = sweep_metrics_df["k"].to_numpy(dtype=float)
    sweep_stability_vals = sweep_metrics_df["stability_score"].to_numpy(dtype=float)
    best_stability = float(np.interp(k_suggested, sweep_k_vals, sweep_stability_vals))

    k_offset_applied = float(k_overdetect_offset)
    while True:
        k_candidate = float(
            np.clip(k_suggested + k_offset_applied, k_hard_min, k_hard_max)
        )
        stability_candidate = float(
            np.interp(k_candidate, sweep_k_vals, sweep_stability_vals)
        )
        if stability_candidate >= (best_stability - stability_guardrail_drop):
            break
        if k_offset_applied >= 0.0:
            break
        k_offset_applied = min(0.0, k_offset_applied + 0.05)

    k_selected = float(np.clip(k_suggested + k_offset_applied, k_hard_min, k_hard_max))
    stability_selected = float(
        np.interp(k_selected, sweep_k_vals, sweep_stability_vals)
    )
else:
    # Fallback when sweep grid is empty after hard-bound clipping.
    k_offset_applied = 0.0
    k_selected = float(np.clip(k_suggested, k_hard_min, k_hard_max))
    best_stability = np.nan
    stability_selected = np.nan

# Run final detection using the guardrail-adjusted selected k for downstream QC/analysis.
selected_events_df, vel_thresh, vel_mean, vel_std = _events_for_k(k_selected)
all_saccades_df = selected_events_df.copy()
upward_saccades_df = all_saccades_df[all_saccades_df["direction"] == "upward"].copy()
downward_saccades_df = all_saccades_df[
    all_saccades_df["direction"] == "downward"
].copy()

print("✅ Final detection re-run complete (using guardrail-adjusted k)")
print(f"  k_suggested={k_suggested:.2f}")
print(
    f"  requested_offset={k_overdetect_offset:+.2f}, applied_offset={k_offset_applied:+.2f}, "
    f"k_selected={k_selected:.2f}"
)
print(
    f"  stability(best={best_stability:.2f}, selected={stability_selected:.2f}, "
    f"guardrail_drop={stability_guardrail_drop:.2f})"
)
print(f"  vel_thresh={vel_thresh:.2f} px/s")
print(
    f"  upward={len(upward_saccades_df)}, downward={len(downward_saccades_df)}, total={len(all_saccades_df)}"
)

In [ ]:
# Cell 6: k-sweep metric curves and interpretation helpers
##########################################################################
fig_k = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=(
        "Number of saccades",
        "Velocity threshold",
        "Median duration",
        "Median amplitude",
        "Stability score",
        "Composite score (selection heuristic)",
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.12,
)


def _add_metric_trace(
    dfm: pd.DataFrame, x_col: str, y_col: str, row: int, col: int, name: str, color: str
):
    fig_k.add_trace(
        go.Scatter(
            x=dfm[x_col],
            y=dfm[y_col],
            mode="lines+markers",
            name=name,
            line=dict(color=color, width=2),
            marker=dict(size=6),
        ),
        row=row,
        col=col,
    )


_add_metric_trace(sweep_metrics_df, "k", "n_saccades", 1, 1, "sweep", "royalblue")

_add_metric_trace(sweep_metrics_df, "k", "vel_thresh", 1, 2, "sweep", "royalblue")

_add_metric_trace(sweep_metrics_df, "k", "median_duration", 2, 1, "sweep", "royalblue")

_add_metric_trace(sweep_metrics_df, "k", "median_amplitude", 2, 2, "sweep", "royalblue")

_add_metric_trace(sweep_metrics_df, "k", "stability_score", 3, 1, "sweep", "royalblue")

if "composite_score" in sweep_metrics_df.columns:
    _add_metric_trace(
        sweep_metrics_df, "k", "composite_score", 3, 2, "sweep", "royalblue"
    )

for r in (1, 2, 3):
    for c in (1, 2):
        fig_k.update_xaxes(title_text="k", row=r, col=c)

fig_k.update_yaxes(title_text="count", row=1, col=1)
fig_k.update_yaxes(title_text="px/s", row=1, col=2)
fig_k.update_yaxes(title_text="seconds", row=2, col=1)
fig_k.update_yaxes(title_text="pixels", row=2, col=2)
fig_k.update_yaxes(title_text="0-100", row=3, col=1)
fig_k.update_yaxes(title_text="a.u.", row=3, col=2)

fig_k.add_vline(x=k_suggested, line_dash="dash", line_color="green")
fig_k.add_vline(x=k_selected, line_dash="dot", line_color="purple")
fig_k.update_layout(
    title=(
        f"k-sweep metrics ({eye_with_least_low_confidence}, eye={selected_eye_code}) | "
        f"suggested k={k_suggested:.2f}, selected k={k_selected:.2f}"
    ),
    template="plotly_white",
    width=900,
    height=650,
    font=dict(size=10),
    title_font=dict(size=14),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    legend_font=dict(size=9),
)
fig_k.update_annotations(font=dict(size=10))
fig_k.update_xaxes(tickfont=dict(size=9), title_font=dict(size=10))
fig_k.update_yaxes(tickfont=dict(size=9), title_font=dict(size=10))
fig_k.show()

print("\nSuggested k-selection approach:")
print("- Prefer k values with high stability_score and smooth count-vs-k behavior.")
print(
    "- This heuristic is intentionally biased toward slight over-detection (lower k / more events)."
)
print("- Single-pass sweep uses the coarse k-range with fine-step resolution.")
print(
    "- Use median duration/amplitude curves as sanity checks; abrupt jumps suggest unstable detection."
)
print(f"- Current heuristic suggestion: k = {k_suggested:.2f}")
print(
    f"- Final selected k applies absolute offset ({k_overdetect_offset:+.2f}) with "
    f"stability guardrail ({stability_guardrail_drop:.1f} points): k = {k_selected:.2f}"
)

In [ ]:
# Cell 7: Post-detection cleanup (transient filter, then amplitude filter)
##########################################################################
all_saccades_df_pre_filter = all_saccades_df.copy()
upward_saccades_df_pre_filter = upward_saccades_df.copy()
downward_saccades_df_pre_filter = downward_saccades_df.copy()

# 1) Same-direction de-dup first (keep strongest in close clusters).
all_saccades_df_after_detection = all_saccades_df_pre_filter.sort_values(
    "time"
).reset_index(drop=True)
all_saccades_df_after_dedup, dedup_removed_saccades_df = _dedupe_same_direction_events(
    all_saccades_df_after_detection,
    min_isi_s=same_direction_dedup_window_s,
)
n_before_dedup = len(all_saccades_df_after_detection)
n_after_dedup = len(all_saccades_df_after_dedup)
n_removed_dedup = len(dedup_removed_saccades_df)
removed_dedup_pct = (
    (100.0 * n_removed_dedup / n_before_dedup) if n_before_dedup > 0 else 0.0
)
print(
    f"✅ Same-direction de-dup applied (window={same_direction_dedup_window_s:.3f}s) | "
    f"kept={n_after_dedup}/{n_before_dedup}, removed={n_removed_dedup} ({removed_dedup_pct:.1f}%)"
)

# 2) Transient-pair filter second (teal markers in Cell 8).
all_saccades_df, transient_removed_saccades_df = _split_transient_pairs(
    all_saccades_df_after_dedup,
    max_isi_s=refractory_period_s,
    max_net_displacement_px=transient_pair_max_net_displacement_px,
)
n_before_transient = len(all_saccades_df_after_dedup)
n_after_transient = len(all_saccades_df)
n_removed_transient = len(transient_removed_saccades_df)
removed_transient_pct = (
    (100.0 * n_removed_transient / n_before_transient)
    if n_before_transient > 0
    else 0.0
)
print(
    f"✅ Transient-pair filter applied (window={refractory_period_s:.3f}s, "
    f"net_disp<={transient_pair_max_net_displacement_px:.2f} px) | "
    f"kept={n_after_transient}/{n_before_transient}, "
    f"removed={n_removed_transient} ({removed_transient_pct:.1f}%)"
)

# 3) Amplitude filter third (gray markers in Cell 8).
all_saccades_df_pre_amplitude = all_saccades_df.copy()
all_saccades_df = all_saccades_df[
    all_saccades_df["amplitude"] >= min_saccade_amplitude_px
].copy()
all_saccades_df = all_saccades_df.sort_values("time").reset_index(drop=True)
upward_saccades_df = all_saccades_df[all_saccades_df["direction"] == "upward"].copy()
downward_saccades_df = all_saccades_df[
    all_saccades_df["direction"] == "downward"
].copy()
removed_saccades_df = all_saccades_df_pre_amplitude[
    all_saccades_df_pre_amplitude["amplitude"] < min_saccade_amplitude_px
].copy()

n_before_amp = len(all_saccades_df_pre_amplitude)
n_after_amp = len(all_saccades_df)
n_removed_amp = len(removed_saccades_df)
removed_amp_pct = (100.0 * n_removed_amp / n_before_amp) if n_before_amp > 0 else 0.0
print(
    f"✅ Amplitude filter applied (min_saccade_amplitude_px={min_saccade_amplitude_px:.2f} px) | "
    f"kept={n_after_amp}/{n_before_amp}, removed={n_removed_amp} ({removed_amp_pct:.1f}%)"
)

if debug and n_removed_amp > 0:
    display(
        removed_saccades_df[
            [
                "time",
                "direction",
                "velocity",
                "duration",
                "amplitude",
                "start_time",
                "end_time",
            ]
        ].head(30)
    )

In [ ]:
# Cell 8: QC plots for selected-k detection (after amplitude filtering)
##########################################################################
if plot_detection_qc:
    print(
        "ℹ️ QC note: In decimated plots, rectangle widths may not always reflect true "
        "saccade duration exactly."
    )
    print("ℹ️ Filled circles: detected saccade points (velocity threshold crossings).")
    print("ℹ️ Teal hollow circles: transient-pair filtered events.")
    print("ℹ️ Gray hollow circles: events dropped by the amplitude threshold.")
    # Fast QC settings
    qc_max_points_global = 30000
    qc_max_points_velocity = 150000
    qc_density_window_s = 30.0
    qc_density_step_s = 10.0
    qc_local_window_s = 30.0

    # 1) Global lightweight overview (decimated traces + event spans/peaks)
    time_rel_s = (df_work["Seconds"] - df_work["Seconds"].iloc[0]).to_numpy()
    n_pts = len(df_work)
    stride_pos = max(1, int(np.ceil(n_pts / qc_max_points_global)))
    stride_vel = max(1, int(np.ceil(n_pts / qc_max_points_velocity)))
    idx_pos = np.arange(0, n_pts, stride_pos)
    idx_vel = np.arange(0, n_pts, stride_vel)

    fig_overlay = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            "X position (decimated)",
            "Velocity with threshold",
        ),
    )

    fig_overlay.add_trace(
        go.Scatter(
            x=time_rel_s[idx_pos],
            y=df_work["X_raw"].to_numpy()[idx_pos],
            mode="lines",
            name=f"X_raw (1/{stride_pos} decimation)",
            line=dict(width=1, color="royalblue"),
        ),
        row=1,
        col=1,
    )
    fig_overlay.add_trace(
        go.Scatter(
            x=time_rel_s[idx_vel],
            y=df_work["vel_x_smooth"].to_numpy()[idx_vel],
            mode="lines",
            name=f"vel_x_smooth (1/{stride_vel} decimation)",
            line=dict(width=1, color="firebrick"),
        ),
        row=2,
        col=1,
    )

    fig_overlay.add_hline(
        y=vel_thresh,
        line_dash="dash",
        line_color="green",
        annotation_text=f"+thr ({vel_thresh:.1f})",
        row=2,
        col=1,
    )
    fig_overlay.add_hline(
        y=-vel_thresh,
        line_dash="dash",
        line_color="green",
        annotation_text=f"-thr ({-vel_thresh:.1f})",
        row=2,
        col=1,
    )

    # Direction-colored threshold-crossing markers on position panel.
    # Use start_time (onset crossing) and draw as dots.
    if len(upward_saccades_df) > 0:
        up_x = upward_saccades_df["start_time"] - df_work["Seconds"].iloc[0]
        up_y = np.interp(
            upward_saccades_df["start_time"].to_numpy(dtype=float),
            df_work["Seconds"].to_numpy(dtype=float),
            df_work["X_raw"].to_numpy(dtype=float),
        )
        fig_overlay.add_trace(
            go.Scatter(
                x=up_x,
                y=up_y,
                mode="markers",
                name="upward threshold-crossing",
                marker=dict(color="limegreen", size=7, symbol="circle"),
            ),
            row=1,
            col=1,
        )
    if len(downward_saccades_df) > 0:
        down_x = downward_saccades_df["start_time"] - df_work["Seconds"].iloc[0]
        down_y = np.interp(
            downward_saccades_df["start_time"].to_numpy(dtype=float),
            df_work["Seconds"].to_numpy(dtype=float),
            df_work["X_raw"].to_numpy(dtype=float),
        )
        fig_overlay.add_trace(
            go.Scatter(
                x=down_x,
                y=down_y,
                mode="markers",
                name="downward threshold-crossing",
                marker=dict(color="mediumpurple", size=7, symbol="circle"),
            ),
            row=1,
            col=1,
        )

    # Threshold-crossings filtered as transient pairs (post-hoc refractory logic).
    # Show as teal hollow circles on position panel only (no duration rectangles).
    if (
        "transient_removed_saccades_df" in globals()
        and len(transient_removed_saccades_df) > 0
    ):
        transient_x = (
            transient_removed_saccades_df["start_time"] - df_work["Seconds"].iloc[0]
        )
        transient_y = np.interp(
            transient_removed_saccades_df["start_time"].to_numpy(dtype=float),
            df_work["Seconds"].to_numpy(dtype=float),
            df_work["X_raw"].to_numpy(dtype=float),
        )
        fig_overlay.add_trace(
            go.Scatter(
                x=transient_x,
                y=transient_y,
                mode="markers",
                name="transient-pair filtered threshold-crossing",
                marker=dict(
                    color="rgba(0,0,0,0)",
                    size=8,
                    symbol="circle",
                    line=dict(color="darkcyan", width=1.5),
                ),
            ),
            row=1,
            col=1,
        )

    # Threshold-crossings that were filtered out by amplitude threshold.
    # Show as gray hollow circles on position panel only (no duration rectangles).
    if "removed_saccades_df" in globals() and len(removed_saccades_df) > 0:
        removed_x = removed_saccades_df["start_time"] - df_work["Seconds"].iloc[0]
        removed_y = np.interp(
            removed_saccades_df["start_time"].to_numpy(dtype=float),
            df_work["Seconds"].to_numpy(dtype=float),
            df_work["X_raw"].to_numpy(dtype=float),
        )
        fig_overlay.add_trace(
            go.Scatter(
                x=removed_x,
                y=removed_y,
                mode="markers",
                name="amplitude-filtered threshold-crossing",
                marker=dict(
                    color="rgba(0,0,0,0)",
                    size=8,
                    symbol="circle",
                    line=dict(color="gray", width=1.5),
                ),
            ),
            row=1,
            col=1,
        )

    # Direction-colored event spans only on position panel (start_time->end_time).
    # Build all shapes first, then assign once (much faster than add_shape in a loop).
    if len(all_saccades_df) > 0:
        t0 = float(df_work["Seconds"].iloc[0])
        starts_rel = all_saccades_df["start_time"].to_numpy(dtype=float) - t0
        ends_rel = all_saccades_df["end_time"].to_numpy(dtype=float) - t0
        dirs = all_saccades_df["direction"].to_numpy()

        overlay_shapes = []
        for x0, x1, direction in zip(starts_rel, ends_rel, dirs):
            is_up = direction == "upward"
            overlay_shapes.append(
                dict(
                    type="rect",
                    x0=float(x0),
                    x1=float(x1),
                    y0=0,
                    y1=1,
                    xref="x",
                    yref="y domain",
                    fillcolor="rgba(34,139,34,0.10)"
                    if is_up
                    else "rgba(128,0,128,0.10)",
                    line=dict(
                        color="rgba(34,139,34,0.25)"
                        if is_up
                        else "rgba(128,0,128,0.25)",
                        width=1,
                    ),
                )
            )
        fig_overlay.update_layout(shapes=overlay_shapes)

    fig_overlay.update_layout(
        title=f"Selected-k detection QC (global, fast) ({eye_with_least_low_confidence}, eye={selected_eye_code})",
        template="plotly_white",
        height=650,
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0.0),
    )
    fig_overlay.update_xaxes(title_text="Relative time (s)", row=2, col=1)
    fig_overlay.update_yaxes(title_text="X position (px)", row=1, col=1)
    fig_overlay.update_yaxes(title_text="Velocity (px/s)", row=2, col=1)
    fig_overlay.show()

    # Note: regime-stratified local windows were removed by request.

    # 2) Distribution panel (single row, 4 columns):
    #    |peak velocity|, amplitude (kept + filtered-out), duration, ISI(log-x).
    if len(all_saccades_df) > 0:
        n_removed_local = (
            len(removed_saccades_df) if "removed_saccades_df" in globals() else 0
        )
        fig_dist = make_subplots(
            rows=1,
            cols=4,
            subplot_titles=(
                "Peak velocity distribution",
                "Amplitude distribution",
                "Duration distribution",
                "ISI distribution (log x-axis)",
            ),
            horizontal_spacing=0.08,
        )

        fig_dist.add_trace(
            go.Histogram(
                x=all_saccades_df["velocity"].abs(),
                nbinsx=60,
                name="|peak velocity| (kept)",
                marker_color="indianred",
                opacity=0.75,
            ),
            row=1,
            col=1,
        )
        fig_dist.add_trace(
            go.Histogram(
                x=all_saccades_df["amplitude"],
                nbinsx=80,
                name=f"Amplitude kept (n={len(all_saccades_df)})",
                marker_color="steelblue",
                opacity=0.6,
            ),
            row=1,
            col=2,
        )
        if n_removed_local > 0:
            fig_dist.add_trace(
                go.Histogram(
                    x=removed_saccades_df["amplitude"],
                    nbinsx=80,
                    name=f"Amplitude filtered out (n={n_removed_local})",
                    marker_color="indianred",
                    opacity=0.7,
                ),
                row=1,
                col=2,
            )
        fig_dist.add_vline(
            x=min_saccade_amplitude_px,
            line_dash="dash",
            line_color="black",
            row=1,
            col=2,
        )

        fig_dist.add_trace(
            go.Histogram(
                x=all_saccades_df["duration"],
                nbinsx=60,
                name="duration (kept)",
                marker_color="mediumpurple",
                opacity=0.75,
            ),
            row=1,
            col=3,
        )

        isi_kept = (
            np.diff(np.sort(all_saccades_df["time"].to_numpy()))
            if len(all_saccades_df) >= 2
            else np.array([])
        )
        isi_kept = isi_kept[isi_kept > 0]

        if len(isi_kept) > 0:
            isi_min = float(np.min(isi_kept))
            isi_max = float(np.max(isi_kept))
            if np.isclose(isi_min, isi_max):
                bins = np.array([isi_min * 0.8, isi_max * 1.2])
            else:
                bins = np.logspace(np.log10(isi_min), np.log10(isi_max), 80)

            counts_kept, edges_kept = np.histogram(isi_kept, bins=bins)
            centers_kept = np.sqrt(edges_kept[:-1] * edges_kept[1:])
            fig_dist.add_trace(
                go.Bar(
                    x=centers_kept,
                    y=counts_kept,
                    width=np.diff(edges_kept),
                    marker_color="darkcyan",
                    opacity=0.85,
                    name=f"ISI kept (n={len(isi_kept)})",
                ),
                row=1,
                col=4,
            )

            fig_dist.update_xaxes(type="log", row=1, col=4)
            fig_dist.update_xaxes(
                range=[np.log10(0.1), np.log10(max(isi_max, 0.1 * 1.01))],
                row=1,
                col=4,
            )
            fig_dist.add_vline(
                x=refractory_period_s,
                line_dash="dash",
                line_color="red",
                row=1,
                col=4,
            )
        else:
            print("ℹ️ ISI panel empty (need at least 2 kept detected saccades).")

        fig_dist.update_layout(
            template="plotly_white",
            height=380,
            width=1650,
            barmode="overlay",
            legend=dict(
                orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0
            ),
        )
        fig_dist.update_xaxes(title_text="|peak velocity| (px/s)", row=1, col=1)
        fig_dist.update_xaxes(title_text="Amplitude (px)", row=1, col=2)
        fig_dist.update_xaxes(title_text="Duration (s)", row=1, col=3)
        fig_dist.update_xaxes(title_text="ISI (s, log scale)", row=1, col=4)
        fig_dist.update_yaxes(title_text="Count", row=1, col=1)
        fig_dist.update_yaxes(title_text="Count", row=1, col=2)
        fig_dist.update_yaxes(title_text="Count", row=1, col=3)
        fig_dist.update_yaxes(title_text="Count", row=1, col=4)
        fig_dist.show()
    else:
        print(
            "ℹ️ Distribution panel skipped (no detected saccades after amplitude filtering)."
        )

In [ ]:
# Cell 9: Manual full-resolution QC windows (3-column layout)
##########################################################################
# Relative-time windows requested for detailed inspection
manual_qc_windows = [
    (162, 164, "overdetection"),
    (216.0, 218.0, "fast transient"),
    (575.0, 576.0, ""),
    (640.0, 645.0, "small saccade filtering"),
    (1031.0, 1033.0, "fast transient"),
    (1305.0, 1310.0, "undetected"),
    (1370.0, 1375.0, "undetected"),
    (1540.0, 1543.0, "undetected"),
]

if plot_detection_qc:
    t0 = float(df_work["Seconds"].iloc[0])
    n_windows = len(manual_qc_windows)
    n_cols = 3
    n_rows = int(np.ceil(n_windows / n_cols))
    n_plot_rows = 2 * n_rows

    # make_subplots assigns subplot titles in row-major order.
    # Build titles with explicit row/col indexing so each panel label matches its data.
    subplot_titles = [""] * (n_plot_rows * n_cols)
    for i, (ws_rel, we_rel, note) in enumerate(manual_qc_windows):
        block_row = i // n_cols
        row_pos = (2 * block_row) + 1
        row_vel = row_pos + 1
        col = i % n_cols + 1
        note_txt = f" ({note})" if note else ""
        base_label = f"{ws_rel:.0f}-{we_rel:.0f}s{note_txt}"
        idx_pos = (row_pos - 1) * n_cols + (col - 1)
        idx_vel = (row_vel - 1) * n_cols + (col - 1)
        subplot_titles[idx_pos] = f"{base_label} - position"
        subplot_titles[idx_vel] = f"{base_label} - velocity"

    # Single paired figure per window: top=position, bottom=velocity
    fig_manual_pairs = make_subplots(
        rows=n_plot_rows,
        cols=n_cols,
        subplot_titles=tuple(subplot_titles),
        horizontal_spacing=0.06,
        vertical_spacing=0.08,
    )

    for i, (ws_rel, we_rel, _) in enumerate(manual_qc_windows):
        block_row = i // n_cols
        row_pos = (2 * block_row) + 1
        row_vel = row_pos + 1
        col = i % n_cols + 1
        ws_abs = t0 + ws_rel
        we_abs = t0 + we_rel

        mask = (df_work["Seconds"] >= ws_abs) & (df_work["Seconds"] <= we_abs)
        if not mask.any():
            continue

        x_rel = (df_work.loc[mask, "Seconds"] - t0).to_numpy()
        x_raw = df_work.loc[mask, "X_raw"].to_numpy()
        vel = df_work.loc[mask, "vel_x_smooth"].to_numpy()

        # Position trace (top panel)
        fig_manual_pairs.add_trace(
            go.Scatter(
                x=x_rel,
                y=x_raw,
                mode="lines",
                line=dict(color="royalblue", width=1),
                name="X_raw",
                showlegend=(i == 0),
            ),
            row=row_pos,
            col=col,
        )

        # Velocity trace (bottom panel)
        fig_manual_pairs.add_trace(
            go.Scatter(
                x=x_rel,
                y=vel,
                mode="lines",
                line=dict(color="firebrick", width=1),
                name="vel_x_smooth",
                showlegend=(i == 0),
            ),
            row=row_vel,
            col=col,
        )

        # Threshold lines (velocity panel)
        fig_manual_pairs.add_hline(
            y=vel_thresh,
            line_dash="dash",
            line_color="green",
            line_width=1,
            row=row_vel,
            col=col,
        )
        fig_manual_pairs.add_hline(
            y=-vel_thresh,
            line_dash="dash",
            line_color="green",
            line_width=1,
            row=row_vel,
            col=col,
        )

        # Keep-event threshold-crossing markers on position panel.
        up_local = upward_saccades_df[
            (upward_saccades_df["start_time"] >= ws_abs)
            & (upward_saccades_df["start_time"] <= we_abs)
        ]
        if len(up_local) > 0:
            up_x = up_local["start_time"] - t0
            up_y = np.interp(
                up_local["start_time"].to_numpy(dtype=float),
                df_work["Seconds"].to_numpy(dtype=float),
                df_work["X_raw"].to_numpy(dtype=float),
            )
            fig_manual_pairs.add_trace(
                go.Scatter(
                    x=up_x,
                    y=up_y,
                    mode="markers",
                    name="upward threshold-crossing",
                    marker=dict(color="limegreen", size=7, symbol="circle"),
                    showlegend=(i == 0),
                ),
                row=row_pos,
                col=col,
            )

        down_local = downward_saccades_df[
            (downward_saccades_df["start_time"] >= ws_abs)
            & (downward_saccades_df["start_time"] <= we_abs)
        ]
        if len(down_local) > 0:
            down_x = down_local["start_time"] - t0
            down_y = np.interp(
                down_local["start_time"].to_numpy(dtype=float),
                df_work["Seconds"].to_numpy(dtype=float),
                df_work["X_raw"].to_numpy(dtype=float),
            )
            fig_manual_pairs.add_trace(
                go.Scatter(
                    x=down_x,
                    y=down_y,
                    mode="markers",
                    name="downward threshold-crossing",
                    marker=dict(color="mediumpurple", size=7, symbol="circle"),
                    showlegend=(i == 0),
                ),
                row=row_pos,
                col=col,
            )

        # Transient-pair filtered events (teal hollow circles) on position panel.
        transient_local = transient_removed_saccades_df[
            (transient_removed_saccades_df["start_time"] >= ws_abs)
            & (transient_removed_saccades_df["start_time"] <= we_abs)
        ]
        if len(transient_local) > 0:
            transient_x = transient_local["start_time"] - t0
            transient_y = np.interp(
                transient_local["start_time"].to_numpy(dtype=float),
                df_work["Seconds"].to_numpy(dtype=float),
                df_work["X_raw"].to_numpy(dtype=float),
            )
            fig_manual_pairs.add_trace(
                go.Scatter(
                    x=transient_x,
                    y=transient_y,
                    mode="markers",
                    name="transient-pair filtered threshold-crossing",
                    marker=dict(
                        color="rgba(0,0,0,0)",
                        size=8,
                        symbol="circle",
                        line=dict(color="darkcyan", width=1.5),
                    ),
                    showlegend=(i == 0),
                ),
                row=row_pos,
                col=col,
            )

        # Amplitude-filtered events (gray hollow circles) on position panel.
        amp_removed_local = removed_saccades_df[
            (removed_saccades_df["start_time"] >= ws_abs)
            & (removed_saccades_df["start_time"] <= we_abs)
        ]
        if len(amp_removed_local) > 0:
            amp_removed_x = amp_removed_local["start_time"] - t0
            amp_removed_y = np.interp(
                amp_removed_local["start_time"].to_numpy(dtype=float),
                df_work["Seconds"].to_numpy(dtype=float),
                df_work["X_raw"].to_numpy(dtype=float),
            )
            fig_manual_pairs.add_trace(
                go.Scatter(
                    x=amp_removed_x,
                    y=amp_removed_y,
                    mode="markers",
                    name="amplitude-filtered threshold-crossing",
                    marker=dict(
                        color="rgba(0,0,0,0)",
                        size=8,
                        symbol="circle",
                        line=dict(color="gray", width=1.5),
                    ),
                    showlegend=(i == 0),
                ),
                row=row_pos,
                col=col,
            )

        # Keep-event duration spans on position panel only.
        # Use overlap condition so spans appear even if peak is just outside window.
        events_local = all_saccades_df[
            (all_saccades_df["start_time"] <= we_abs)
            & (all_saccades_df["end_time"] >= ws_abs)
        ]
        if len(events_local) > 0:
            for _, ev in events_local.iterrows():
                x0 = max(ws_abs, ev["start_time"]) - t0
                x1 = min(we_abs, ev["end_time"]) - t0
                fill = (
                    "rgba(34,139,34,0.10)"
                    if ev["direction"] == "upward"
                    else "rgba(128,0,128,0.10)"
                )
                line = (
                    "rgba(34,139,34,0.25)"
                    if ev["direction"] == "upward"
                    else "rgba(128,0,128,0.25)"
                )
                fig_manual_pairs.add_vrect(
                    x0=x0,
                    x1=x1,
                    fillcolor=fill,
                    line=dict(color=line, width=1),
                    row=row_pos,
                    col=col,
                )

    fig_manual_pairs.update_layout(
        title="Manual full-resolution QC windows: Paired position/velocity",
        template="plotly_white",
        height=260 * n_plot_rows,
        width=1400,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
        font=dict(size=10),
    )

    for rr in range(1, n_plot_rows + 1):
        for cc in range(1, n_cols + 1):
            if rr % 2 == 0:
                fig_manual_pairs.update_xaxes(
                    title_text="Relative time (s)",
                    row=rr,
                    col=cc,
                    tickfont=dict(size=9),
                    title_font=dict(size=10),
                )
            else:
                fig_manual_pairs.update_xaxes(
                    showticklabels=False,
                    row=rr,
                    col=cc,
                )

            fig_manual_pairs.update_yaxes(
                title_text="X position (px)" if rr % 2 == 1 else "Velocity (px/s)",
                row=rr,
                col=cc,
                tickfont=dict(size=9),
                title_font=dict(size=10),
            )
    fig_manual_pairs.show()

In [ ]:
# Cell 10: Manual QC count entry (user-reported misses/overdetections)
##########################################################################
existing_manual_qc = metadata.get("manual_qc_counts", {})
default_missed_saccades = int(existing_manual_qc.get("manual_missed_saccades", 0))
default_false_positives = int(existing_manual_qc.get("manual_false_positives", 0))

try:
    import ipywidgets as widgets

    header = widgets.HTML(
        value="<b>Manual QC entry</b>: enter count of missed true saccades and overdetections."
    )
    missed_input = widgets.BoundedIntText(
        value=max(0, default_missed_saccades),
        min=0,
        step=1,
        description="Missed drops:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="320px"),
    )
    false_pos_input = widgets.BoundedIntText(
        value=max(0, default_false_positives),
        min=0,
        step=1,
        description="Overdetect.:",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="320px"),
    )
    save_button = widgets.Button(
        description="Save QC counts to metadata",
        button_style="success",
        icon="save",
        layout=widgets.Layout(width="320px"),
    )
    status = widgets.HTML()

    def _save_manual_qc_counts(_):
        metadata["manual_qc_counts"] = {
            "manual_missed_saccades": int(missed_input.value),
            "manual_false_positives": int(false_pos_input.value),
            "updated_utc": pd.Timestamp.utcnow().isoformat(),
        }
        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=2)
        status.value = (
            "✅ Saved manual QC counts: "
            f"missed={int(missed_input.value)}, "
            f"overdetections={int(false_pos_input.value)}"
        )

    save_button.on_click(_save_manual_qc_counts)
    display(widgets.VBox([header, missed_input, false_pos_input, save_button, status]))
except Exception:
    print("⚠️ ipywidgets unavailable. Manual QC widget could not be displayed.")